# Plot Ice Thickness × Concentration Along Route (Unstructured Grid)

This notebook creates contour maps of the product of sea ice thickness and
concentration (= `iceVolumeCell`, the effective ice thickness in metres) and overlays:
1. Route points (transect of interest)
2. MPAS-SI mesh points used for interpolation

The Median, 5th, and 95th percentile fields are selectable via `thkconc_statistic`.
Contouring is performed directly on the native MPAS unstructured mesh in
NorthPolarStereo projected coordinates to avoid antimeridian and regridding artefacts.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import xarray
import os
from scipy.spatial import KDTree

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as mtri
import cmocean

In [ ]:
# Configuration and parameters
years  = ['2000']
months = ['05']

# Statistic to plot: 'median', '5th', or '95th'
thkconc_statistic = '95th'  # Options: 'median', '5th', '95th'

# File paths
pathBase     = '/Users/stephenprice/projects/CINS/ColdHarbor/test-data/v3.LR.historical_thkConc_ensembleStats'
prefix       = 'v3.LR.historical_EnsStats'
mpassiPrefix = '.mpassi.hist.am.timeSeriesStatsDaily.'

# Map output path
mapPath = '/Users/stephenprice/projects/CINS/ColdHarbor/maps'

# Conversion factor
rad2deg = 180.0 / np.pi

In [ ]:
# Load MPAS-O mesh lat/lon coordinates
os.chdir(pathBase)
fileIn = '../mpaso-IcoswISC30E3r5-restart.nc'
ds = xarray.open_dataset(fileIn)

lat_mesh = ds.latCell.values * rad2deg
lon_mesh = ds.lonCell.values * rad2deg

# Convert longitudes to -180 to 180
lon_mesh_temp = np.zeros(np.shape(lon_mesh))
for i in np.arange(0, np.size(lon_mesh)):
    if lon_mesh[i] > 180:
        lon_mesh_temp[i] = lon_mesh[i] - 360
    else:
        lon_mesh_temp[i] = lon_mesh[i]
lon_mesh = lon_mesh_temp
del lon_mesh_temp

# Filter to Northern region (>= 60 deg N)
ind = np.where(lat_mesh >= 60.0)[0]
lat_mesh = lat_mesh[ind]
lon_mesh = lon_mesh[ind]

print(f'Loaded {len(lat_mesh)} mesh points')

In [ ]:
# Load route track data
fileIn = '../arctic_route_sections.nc'
ds = xarray.open_dataset(fileIn)

# Extract route sections (using EJCD route combination)
lat_E = ds.lat_E.values; lon_E = ds.lon_E.values; dis_E = ds.dis_E.values
lat_J = ds.lat_J.values; lon_J = ds.lon_J.values; dis_J = ds.dis_J.values
lat_C = ds.lat_C.values; lon_C = ds.lon_C.values; dis_C = ds.dis_C.values
lat_D = ds.lat_D.values; lon_D = ds.lon_D.values; dis_D = ds.dis_D.values

# Combine route sections into single transect
trans_lat  = np.hstack((lat_E, lat_J, lat_C, lat_D))
trans_lon  = np.hstack((lon_E, lon_J, lon_C, lon_D))
trans_dist = np.hstack((dis_E, dis_J, dis_C, dis_D))

print(f'Loaded route with {len(trans_lat)} points')

In [ ]:
# Find nearest neighbor mesh points for each route point
tree = KDTree(list(zip(lon_mesh.flatten(), lat_mesh.flatten())))

kneighbor = 4
d, inds = tree.query(list(zip(trans_lon, trans_lat)), k=kneighbor)

print(f'Found {kneighbor} nearest neighbors for each of {len(trans_lat)} route points')

In [ ]:
# Load ice thickness and concentration ensemble statistics
os.chdir(pathBase)

year  = years[0]
month = months[0]

fileIn = prefix + mpassiPrefix + year + '-' + month + '-01.nc'
print(f'Reading data from: {fileIn}')
dataIn = xarray.open_dataset(fileIn)

# Select the matching percentile for both iceAreaCell and iceVolumeCell
if thkconc_statistic == 'median':
    iceArea   = dataIn.timeDaily_avg_iceAreaCell_ensembleMedian.values
    iceVolume = dataIn.timeDaily_avg_iceVolumeCell_ensembleMedian.values
    stat_label = 'Median'
elif thkconc_statistic == '5th':
    iceArea   = dataIn.timeDaily_avg_iceAreaCell_ensemble5th.values
    iceVolume = dataIn.timeDaily_avg_iceVolumeCell_ensemble5th.values
    stat_label = '5th Percentile'
elif thkconc_statistic == '95th':
    iceArea   = dataIn.timeDaily_avg_iceAreaCell_ensemble95th.values
    iceVolume = dataIn.timeDaily_avg_iceVolumeCell_ensemble95th.values
    stat_label = '95th Percentile'
else:
    raise ValueError(f"Invalid statistic: {thkconc_statistic}. Choose 'median', '5th', or '95th'")

# Thickness x concentration = iceVolumeCell (effective thickness, metres)
# In MPAS-SI: iceVolumeCell = concentration * actual_thickness by convention

# this is the average value of thickness in a grid cell
thkConcData = iceVolume.copy()

# this is the max value of thickness in a grid cell (i.e., the mean thickness for a concentration < 1 implies
# that for the area where that concentration is non-zero, the thickness must be greater than the mean)
# thkConcData = iceVolume / iceArea

# this is the value of thickness weighted by the concentration
# thkConcData = iceVolume * iceArea

ndays  = np.size(thkConcData, axis=0)
ncells = np.size(thkConcData, axis=1)

print(f'Loaded {stat_label} thickness x concentration: {ndays} days, {ncells} cells')

In [ ]:
# Define plotting function using unstructured contouring
def plot_thkconc_unstructured(lon_mesh, lat_mesh, thkConc, trans_lon, trans_lat,
                              inds, stat_label='Median',
                              time_index=15, extent=(-180, 180, 65, 90),
                              vmin=0.0, vmax=3.0, level_step=0.25,
                              cmap=cmocean.cm.ice, max_edge_m=500_000,
                              figsize=(20, 20), dpi=300):
    """
    Create contour map of ice thickness x concentration with route and mesh overlays.

    Contouring is performed on the native MPAS unstructured mesh projected into
    NorthPolarStereo space (metres), avoiding antimeridian artefacts.

    Parameters:
    -----------
    lon_mesh, lat_mesh : array-like
        Mesh coordinates (unstructured, degrees)
    thkConc : array-like
        Thickness x concentration values, shape (ndays, ncells)
    trans_lon, trans_lat : array-like
        Route coordinates (degrees)
    inds : array-like
        Nearest neighbor mesh point indices for each route point
    stat_label : str
        Label for the statistic being plotted
    time_index : int
        Day index to plot (default: 15)
    extent : tuple
        Map extent (lon_min, lon_max, lat_min, lat_max) in degrees
    vmin, vmax : float
        Color scale limits (metres); values below vmin render white
    level_step : float
        Contour level spacing (metres)
    cmap : colormap
        Matplotlib colormap
    max_edge_m : float
        Maximum triangle edge length (metres) for sliver-triangle masking
    figsize, dpi : tuple, int
        Figure size and resolution

    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    noProj  = ccrs.PlateCarree()
    mapProj = ccrs.NorthPolarStereo(central_longitude=0)

    # 1D data arrays
    lon1 = np.asarray(lon_mesh).ravel()
    lat1 = np.asarray(lat_mesh).ravel()
    z1   = np.asarray(thkConc)[time_index, :].ravel()

    # Drop NaNs
    good = np.isfinite(lon1) & np.isfinite(lat1) & np.isfinite(z1)
    lon1, lat1, z1 = lon1[good], lat1[good], z1[good]

    # Project lon/lat -> metres in NorthPolarStereo
    pts = mapProj.transform_points(noProj, lon1, lat1)
    xp, yp = pts[:, 0], pts[:, 1]

    # Drop any points that fail to project
    good_proj = np.isfinite(xp) & np.isfinite(yp)
    xp, yp, z1 = xp[good_proj], yp[good_proj], z1[good_proj]

    # Build Delaunay triangulation in projected (metric) space
    tri = mtri.Triangulation(xp, yp)

    # Mask large sliver triangles at the domain boundary
    verts = tri.triangles
    x0, x1, x2 = xp[verts[:, 0]], xp[verts[:, 1]], xp[verts[:, 2]]
    y0, y1, y2 = yp[verts[:, 0]], yp[verts[:, 1]], yp[verts[:, 2]]
    e0 = np.hypot(x1 - x0, y1 - y0)
    e1 = np.hypot(x2 - x1, y2 - y1)
    e2 = np.hypot(x0 - x2, y0 - y2)
    tri.set_mask(np.max(np.column_stack([e0, e1, e2]), axis=1) > max_edge_m)

    # Figure / axes
    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax  = plt.axes(projection=mapProj)
    ax.set_extent(extent, crs=noProj)
    ax.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                 color='w', linestyle=':', zorder=50)

    # Filled contours - no transform= because data is already in projection coords
    clevels = np.arange(vmin, vmax + 0.5 * level_step, level_step)
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_under('white')
    cf = ax.tricontourf(tri, z1, levels=clevels, cmap=cmap_obj,
                        extend='both', zorder=5)

    cbar = plt.colorbar(cf, ax=ax, shrink=0.75, pad=0.03, orientation='vertical')
    cbar.set_label(f'Thickness x Concentration {stat_label} [m]', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    # Route points (red) - in lon/lat so need transform
    ax.plot(np.asarray(trans_lon), np.asarray(trans_lat),
            linestyle='None', marker='.', markersize=3, color='red',
            transform=noProj, zorder=60, label='Route points')

    # Mesh interpolation points (black)
    inds_use  = np.asarray(inds)[:, 1:]
    flat_inds = np.unique(inds_use.ravel())
    ax.plot(np.asarray(lon_mesh)[flat_inds],
            np.asarray(lat_mesh)[flat_inds],
            linestyle='None', marker='.', markersize=0.5,
            markerfacecolor='none', color='black',
            transform=noProj, zorder=61, label='Mesh points')

    # Land / coastlines on top
    ax.add_feature(cfeature.LAND,      facecolor='0.85', edgecolor='none', zorder=100)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6,                      zorder=101)
    ax.add_feature(cfeature.LAKES,     facecolor='0.85', edgecolor='none', zorder=100)

    ax.set_title(
        f'Ice Thickness x Concentration ({stat_label}), Day: {time_index+1}',
        fontsize=16, weight='bold')

    return fig, ax

In [ ]:
# Create the plot
time_index = 15  # Day 16 of the month
extent = (-180, 180, 65, 90)  # Arctic region

fig, ax = plot_thkconc_unstructured(
    lon_mesh=lon_mesh,
    lat_mesh=lat_mesh,
    thkConc=thkConcData,
    trans_lon=trans_lon,
    trans_lat=trans_lat,
    inds=inds,
    stat_label=stat_label,
    time_index=time_index,
    extent=extent,
    vmin=0.0,
    vmax=3.0,
    level_step=0.25,
    cmap=cmocean.cm.ice,
    figsize=(20, 20),
    dpi=150  # Reduced for faster display
)

plt.show()

In [ ]:
# Optional: Save figure to file
os.chdir(mapPath)
output_filename = f'thkConc_map_{thkconc_statistic}_{year}-{month}_day{time_index+1}.png'
fig.savefig(output_filename, bbox_inches='tight', dpi=300)
print(f'Figure saved to: {mapPath}/{output_filename}')

## Customization Options

### Statistic
Change `thkconc_statistic` in the configuration cell:
- `'median'` — Ensemble median
- `'5th'`    — 5th percentile (low ice conditions)
- `'95th'`   — 95th percentile (heavy ice conditions)

### Time Period
- `years`  — e.g. `['2000']` or `['2025']` (available: 2000, 2025)
- `months` — e.g. `['05']` for May
- `time_index` — day within the month (0-based)

### Route Selection
Modify the route combination in the 'Load route track data' cell.
Current: sections E, J, C, D. Alternative example: A, B, I, H.

### Map Appearance
In the plotting call:
- `extent`     — map boundaries
- `vmin/vmax`  — color scale limits (metres); values below `vmin` render white
- `level_step` — contour interval
- `cmap`       — color scheme (e.g. `cmocean.cm.ice`, `cmocean.cm.deep`)
- `figsize`, `dpi` — figure size and resolution

### Notes on Variables
- `iceVolumeCell` (MPAS-SI) = concentration × thickness = effective ice thickness [m]
- `iceAreaCell`   (MPAS-SI) = ice concentration (fraction 0–1)
- The plotted quantity (thickness × concentration) is therefore `iceVolumeCell` directly.